# 01 · Een sensor kiezen, verbinden en uitlezen (Dragino via Antratek)

**Casus.** Een varkenshouder wil in drie afdelingen van een vleesvarkensstal elke 10 minuten temperatuur en relatieve vochtigheid meten, op batterij, minstens 5 jaar zonder onderhoud, en de metingen op zijn telefoon zien. Er is geen netwerkkabel in de stal, wel wifi en 4G op het erf.

In dit notebook doorlopen we de hele keten zoals je ze in de praktijk zou doorlopen:

1. een sensor kiezen in de webshop (**Antratek**, Belgische/Nederlandse distributeur);
2. in de **handleiding van de fabrikant (Dragino)** terugvinden wat je nodig hebt om de sensor aan de praat te krijgen;
3. de **ruwe bytes** die de sensor via LoRaWAN verstuurt met de hand decoderen, en dat controleren met de officiële decoder;
4. dezelfde sensor in zijn **NB-IoT-variant** bekijken, waar je zelf het protocol (CoAP, UDP, MQTT, TCP) kiest.

> Alle webadressen en sectienummers zijn gecontroleerd op 19 oktober 2026. Websites veranderen; zoek bij twijfel op de productnaam.

## Stap 1 · De webshop: Antratek

Open <https://www.antratek.be/lora/sensor-nodes/temperature-humidity> (categorie *LoRa ▸ Sensor nodes ▸ Temperatuur & vochtigheid*). Je ziet een 25-tal LoRaWAN-sensoren, de meeste van **Dragino**, tussen € 50 en € 300.

Klik op **LSN50v2-S31 – LoRaWAN Temperature & Humidity Sensor** (of de nieuwere **S31-LB**). Lees de Nederlandstalige productbeschrijving en zoek deze gegevens op:

| Wat je zoekt | Waar het staat | Waarom het telt |
|---|---|---|
| Artikelnummer `LSN50v2-S31-EU868` | bovenaan | **EU868** = de Europese LoRaWAN-frequentieband. Een US915-versie werkt hier niet en is zelfs verboden. |
| Welke meetsensor zit erin? | beschrijving: *Sensirion SHT31* | Industriële, gekalibreerde sensor: ±0,2 °C, ±2 % RV (zie handleiding §1.3). |
| Batterij | *8500 mAh Li-SOCl2, tot 10 jaar* | Haalt de eis van 5 jaar; de werkelijke duur hangt af van het zendinterval. |
| Sleutels | *"vooraf geladen met unieke sleutels … OTAA … registreren bij TTN"* | Je hebt een **LoRaWAN-netwerkserver** nodig (bv. The Things Network) en een **gateway** in bereik. |
| Prijs | € 52,95 excl. btw (10 stuks: −10 %) | Drie afdelingen = drie sensoren ≈ € 160. |
| Nieuwere versie | *S31-LB* (zelfde prijs), *S31-LS* (zonnepaneel, € 57,95) | -LB = batterij, -LS = solar. |

**Wat ontbreekt nog?** Een LoRaWAN-sensor praat niet rechtstreeks met het internet. Je hebt een **gateway** nodig (Dragino LPS8v2 of een andere, ethernet/wifi/4G naar het internet), tenzij er al dekking is van een publiek LoRaWAN-netwerk (The Things Network-community, of een betalend netwerk zoals Proximus MyThings). Een gateway kost een veelvoud van een sensor; bij drie afdelingen is dat toch de normale keuze.

**Alternatief zonder gateway:** dezelfde sensor bestaat als **S31-NB** (NB-IoT) en **S31-CB** (NB-IoT én LTE-M). Die verstuurt via het 4G-netwerk, met een simkaart en een (klein) abonnement per sensor. Zie stap 4.

### Opdracht 1.1
Noteer voor de casus je keuze (LoRaWAN of NB-IoT, -LB of -LS) en som op wat je naast de drie sensoren nog moet kopen of regelen.

<details>
<summary><b>Oplossing tonen</b></summary>

Een verdedigbare keuze: **3 × S31-LB (LoRaWAN, batterij)** à € 52,95 plus **één gateway** (bv. Dragino LPS8v2) die via de wifi op het erf aan het internet hangt. Verder nodig: een gratis TTN-account (netwerkserver), de decoder van Dragino, en een dashboard (Datacake of ThingsBoard hebben gratis instapniveaus). Totaal eenmalig ± € 160 sensoren + gateway, geen maandelijkse kosten.

Alternatief: **3 × S31-CB (NB-IoT/LTE-M)**, geen gateway, maar drie simkaarten met abonnement (enkele euro's per maand per sensor) en je moet een server/broker kiezen. Interessant als er geen wifi of stroom voor een gateway is, of bij maar één sensor.

-LS (zonnepaneel) is in een donkere stal zinloos; -LB (batterij) volstaat: bij 10 min interval meerdere jaren (handleiding §4).

</details>

## Stap 2 · De handleiding: Dragino

Antratek verwijst naar de fabrikant. Dragino heeft een documentatiecentrum: <https://wiki.dragino.com/docs/LoRaWAN-End-Node/> ▸ *Temperature & Humidity Sensors* ▸ **S31/S31B-LB / LS**:
<https://wiki.dragino.com/docs/LoRaWAN-End-Node/temperature-humidity-sensors/s31-lb/>

Elke Dragino-handleiding heeft dezelfde opbouw. Dit zijn de secties die je nodig hebt om het toestel **te verbinden en uit te lezen**:

| Sectie | Titel | Wat je eruit haalt |
|---|---|---|
| 1.3 | Specification | meetbereik −40…80 °C, nauwkeurigheid, spanning |
| 1.4 | Sleep mode and working mode | de sensor slaapt en wordt enkel wakker om te meten en te zenden (daarom gaat de batterij jaren mee) |
| 1.6 | BLE connection | configuratie via **Bluetooth** met een app — het tweede verbindingstype in hetzelfde toestel |
| **2.2** | **Quick guide to connect to LoRaWAN server (OTAA)** | stap 1: DevEUI/AppKey van de sticker ingeven in TTN; stap 2: decoder toevoegen; stap 3: knop 5 s indrukken → groene led → *join* |
| **2.3** | **Uplink Payload** | hoe de 11 bytes van elke meting zijn opgebouwd (FPORT=2) |
| **2.4** | **Payload Decoder file** | link naar de officiële decoder op GitHub |
| 2.6 | Temperature Alarm Feature | alarm-uplink bij overschrijden van een drempel |
| 3 | Configure (AT-commando's) | `AT+TDC` zendinterval, `AT+SHTEMP` alarmdrempel, … |
| 4 | Battery & Power Consumption | batterijduur in functie van het zendinterval |

### Checklist "koopklaar" voor een LoRaWAN-sensor

- [ ] juiste **frequentieband** (EU868)
- [ ] **DevEUI, AppEUI/JoinEUI en AppKey** (sticker op het toestel of in de doos)
- [ ] een **LoRaWAN-netwerkserver** (TTN is gratis voor kleine toepassingen; ChirpStack is zelf te hosten)
- [ ] een **gateway** in bereik, met internet via ethernet, wifi of 4G
- [ ] de **payload-decoder** van de fabrikant (anders zie je enkel hex)
- [ ] een plek waar de data naartoe gaat: dashboard (Datacake, ThingsBoard, Node-RED) of eigen software via **MQTT**/**HTTP**-integratie van de netwerkserver

### Opdracht 2.1
Open de handleiding en zoek in §2.2 op wat er gebeurt met de groene led bij een geslaagde *join*. Zoek in §3 het AT-commando om het zendinterval op 10 minuten te zetten (tip: de waarde staat in seconden).

<details>
<summary><b>Oplossing tonen</b></summary>

§2.2 stap 3: na het indrukken van de knop (5 s) knippert de groene led 5× snel, het toestel probeert te joinen, en **na een geslaagde join brandt de groene led 5 s continu**. Daarna begint het uplinks te sturen.

§3.3.1 *Set Transmit Interval Time*: `AT+TDC=600` (waarde in seconden; standaard 1200 = 20 min). Hetzelfde kan via een downlink vanuit TTN: `01 00 02 58` (0x000258 = 600).

</details>

## Stap 3 · De payload met de hand decoderen

Een LoRaWAN-bericht is klein (vaak 11 bytes) en bevat **geen** tekst of JSON: enkel ruwe getallen. Sectie 2.3.2 van de handleiding beschrijft de opbouw van een gewone meting (verzonden op **FPORT = 2**):

| Bytes | 0–1 | 2–5 | 6 | 7–8 | 9–10 |
|---|---|---|---|---|---|
| Inhoud | Batterij (mV) | Unix-timestamp (s) | Alarmvlag & modus & PA8 | Temperatuur (×10, signed) | RV (×10) |

Drie dingen moet je weten om dit te lezen:

1. **Hexadecimaal.** Eén byte = 8 bits = twee hex-tekens (`00`…`FF`, dus 0…255).
2. **Big-endian.** Bij getallen van meerdere bytes staat de *meest* betekenisvolle byte eerst: `0B45` = 0x0B × 256 + 0x45 = 11 × 256 + 69 = **2885**.
3. **Two's complement.** Temperatuur kan negatief zijn. Als het hoogste bit van de 16 bits 1 is (hex-waarde ≥ 0x8000), trek je 65536 af: `FF3F` = 65343 → 65343 − 65536 = −193 → **−19,3 °C**. De handleiding beschrijft dit letterlijk zo.

De handleiding geeft deze voorbeelden: batterij `0B45` → 2885 mV; temperatuur `0105` → 26,1 °C en `FF3F` → −19,3 °C; vochtigheid `0197` → 407 → 40,7 %. We bouwen een payload die precies die voorbeelden bevat en decoderen ze stap voor stap.

In [ ]:
# De voorbeeldpayload (11 bytes = 22 hex-tekens), opgebouwd met de voorbeeldwaarden uit de handleiding
payload_hex = "0B456AD703600001050197"

payload = bytes.fromhex(payload_hex)         # hex-tekst -> bytes
print("Aantal bytes:", len(payload))
for i, b in enumerate(payload):
    print(f"byte {i:2d}: hex {b:02X}  = decimaal {b:3d}  = binair {b:08b}")

### 3a · Batterij (bytes 0–1)

In [ ]:
batterij_mv = payload[0] * 256 + payload[1]          # big-endian: eerste byte weegt 256× zwaarder
# hetzelfde met een ingebouwde functie:
batterij_mv_2 = int.from_bytes(payload[0:2], byteorder="big")

print(f"bytes 0-1 = {payload[0]:02X} {payload[1]:02X} -> {batterij_mv} mV = {batterij_mv/1000:.3f} V")
assert batterij_mv == batterij_mv_2 == 2885           # voorbeeld uit de handleiding

### 3b · Tijdstip (bytes 2–5)

Een *Unix-timestamp* is het aantal seconden sinds 1 januari 1970, 00:00 UTC. De sensor heeft een interne klok die bij de *join* door de netwerkserver wordt gelijkgezet.

In [ ]:
from datetime import datetime, timezone

timestamp = int.from_bytes(payload[2:6], byteorder="big")
tijd_utc = datetime.fromtimestamp(timestamp, tz=timezone.utc)
print(f"bytes 2-5 = {payload[2:6].hex().upper()} -> {timestamp} s -> {tijd_utc:%Y-%m-%d %H:%M:%S} UTC")

### 3c · Vlaggenbyte (byte 6)

Eén byte bevat hier **drie** stukjes informatie, elk op eigen bits. Dat is typisch voor IoT: elke bit telt, want zenden kost energie.

| bit 7 | bits 6–2 | bit 1 | bit 0 |
|---|---|---|---|
| niveau van pin PA8 (1 = laag) | modus (0 = gewone meting, 31 = antwoord op een poll) | — | alarmvlag (1 = alarm) |

Om een bit te lezen gebruik je een *bitmasker* met `&` (EN) en verschuiven met `>>`.

In [ ]:
vlaggen = payload[6]
alarm   = vlaggen & 0x01            # masker 0000 0001
modus   = (vlaggen & 0x7C) >> 2     # masker 0111 1100, daarna 2 posities naar rechts
pa8     = (vlaggen & 0x80) >> 7     # masker 1000 0000

print(f"byte 6 = {vlaggen:02X} = {vlaggen:08b}")
print("alarm :", "JA" if alarm else "nee")
print("modus :", modus, "(0 = gewone meting)")
print("PA8   :", "laag" if pa8 else "hoog")

### 3d · Temperatuur (bytes 7–8) — let op het teken

In [ ]:
ruw = int.from_bytes(payload[7:9], byteorder="big")
if ruw & 0x8000:                     # hoogste bit gezet -> negatief getal
    ruw -= 65536
temperatuur = ruw / 10
print(f"bytes 7-8 = {payload[7:9].hex().upper()} -> {ruw} -> {temperatuur} °C")

# Python kan dit ook rechtstreeks met signed=True:
assert int.from_bytes(payload[7:9], "big", signed=True) / 10 == temperatuur

# Controleer het negatieve voorbeeld uit de handleiding: FF3F -> -19.3
print("FF3F ->", int.from_bytes(bytes.fromhex("FF3F"), "big", signed=True) / 10, "°C")

### 3e · Relatieve vochtigheid (bytes 9–10)

In [ ]:
vochtigheid = int.from_bytes(payload[9:11], byteorder="big") / 10
print(f"bytes 9-10 = {payload[9:11].hex().upper()} -> {vochtigheid} % RV")

### 3f · Alles samen in één functie

Dit is precies wat een *payload-decoder* op de netwerkserver doet: bytes in, betekenisvolle waarden uit.

In [ ]:
def decodeer_s31lb(payload_hex: str, fport: int = 2) -> dict:
    '''Decodeer een S31-LB-uplink (FPORT 2) volgens Dragino-handleiding sectie 2.3.2.'''
    b = bytes.fromhex(payload_hex)
    if fport != 2 or len(b) != 11:
        raise ValueError("Verwacht 11 bytes op FPORT 2")
    return {
        "batterij_V":   int.from_bytes(b[0:2], "big") / 1000,
        "tijd_utc":     datetime.fromtimestamp(int.from_bytes(b[2:6], "big"), tz=timezone.utc).isoformat(),
        "alarm":        bool(b[6] & 0x01),
        "modus":        (b[6] & 0x7C) >> 2,
        "temperatuur_C": int.from_bytes(b[7:9], "big", signed=True) / 10,
        "RV_pct":       int.from_bytes(b[9:11], "big") / 10,
    }

decodeer_s31lb(payload_hex)

### Opdracht 3.1 · Decodeer zelf

Hieronder drie uplinks zoals ze in de TTN-console zouden verschijnen. Decodeer ze **eerst met de hand** (pen en papier, of de rekenmachine van je laptop in programmeermodus), en controleer daarna met de functie.

| | Payload (hex) | Context |
|---|---|---|
| A | `0BC46AD703600000D60278` | afdeling 3, gewone meting |
| B | `0AE6696486D800FFCE0393` | buitensensor in januari |
| C | `0B866A4671380101400320` | afdeling 1, juli |

Vragen: Wat is de temperatuur en RV in elk geval? Welke uplink is een alarm, en waarom zou de sensor dat sturen? Welke sensor heeft de minst volle batterij?

In [ ]:
# Opdracht 3.1 — controle
for naam, p in {"A": "0BC46AD703600000D60278", "B": "0AE6696486D800FFCE0393", "C": "0B866A4671380101400320"}.items():
    print(naam, decodeer_s31lb(p))

<details>
<summary><b>Oplossing tonen</b></summary>

| | Batterij | Tijd (UTC) | Alarm | T (°C) | RV (%) |
|---|---|---|---|---|---|
| A | 3,012 V | 2026-10-20 06:00 | nee | 21,4 | 63,2 |
| B | 2,790 V | 2026-01-12 05:30 | nee | **−5,0** (`FFCE` = 65486 → −50) | 91,5 |
| C | 2,950 V | 2026-07-02 14:10 | **ja** (byte 6 = `01`) | 32,0 | 80,0 |

C is een alarm-uplink: de temperatuur overschrijdt de ingestelde drempel (hittestress in juli); de sensor stuurt dan onmiddellijk, buiten het normale interval. B heeft de laagste batterijspanning (2,79 V), mede omdat een Li-SOCl₂-cel bij vrieskou tijdelijk een lagere spanning geeft.

Veelgemaakte fouten: little-endian lezen (`450B` i.p.v. `0B45`), het teken vergeten bij B, de vlaggenbyte decimaal lezen, niet door 10 delen.

</details>

## Stap 3g · Controle met de officiële decoder van Dragino

Sectie 2.4 van de handleiding verwijst naar <https://github.com/dragino/dragino-end-node-decoder>. Daar staat per product een JavaScript-bestand dat je in TTN (*Payload formatters ▸ Custom JavaScript*) plakt. We halen het bestand op en voeren het uit met Node.js (in Codespaces aanwezig) om onze Python-functie te controleren.

Lees het JavaScript even mee: herken je `bytes[0]<<8 | bytes[1]` (big-endian), `bytes[7]<<24>>16` (truc voor het teken) en `(bytes[6] & 0x7C)>>2` (de modusbits)?

In [ ]:
import requests, subprocess, json, shutil, textwrap

url_decoder = ("https://raw.githubusercontent.com/dragino/dragino-end-node-decoder/main/"
               "S31-LB%26S31B-LB/S31-LB_v1.3_TTN_decoder.txt")
js = requests.get(url_decoder, timeout=15).text
print(js[js.find("function Decoder"):][:900])     # toon het begin van de decoderfunctie

In [ ]:
if shutil.which("node") is None:
    print("Node.js niet gevonden — sla deze controle over of vergelijk de code handmatig.")
else:
    bytes_js = ", ".join(str(b) for b in bytes.fromhex(payload_hex))
    script = js + f"\nconsole.log(JSON.stringify(Decoder([{bytes_js}], 2)));\n"
    uit = subprocess.run(["node", "-e", script], capture_output=True, text=True, timeout=20)
    resultaat_js = json.loads(uit.stdout)
    print("Officiële decoder :", resultaat_js)
    print("Onze Python-functie:", decodeer_s31lb(payload_hex))
    assert resultaat_js["TempC_SHT31"] == decodeer_s31lb(payload_hex)["temperatuur_C"]
    assert resultaat_js["Hum_SHT31"]   == decodeer_s31lb(payload_hex)["RV_pct"]
    print("\nBeide geven dezelfde temperatuur en RV. (Data_time staat in de lokale tijdzone van Node.)")

## Stap 4 · Dezelfde sensor via NB-IoT/LTE-M: jij kiest het protocol

Handleiding S31-CB: <https://wiki.dragino.com/docs/NB-IoT/temperature-humidity-sensors/s31-cb/>

Hier zit geen LoRaWAN-radio in, maar een 4G-module (Quectel BG95) met een nanosimkaart. De sensor verbindt **zelf** met het internet en stuurt de meting rechtstreeks naar een server van jouw keuze. Sectie 2.1–2.2 van de handleiding laat zien dat je met één AT-commando het **protocol** én de **payloadvorm** kiest:

```
AT+PRO=1,0   // CoAP, hex-payload
AT+PRO=1,5   // CoAP, JSON-payload
AT+PRO=2,5   // UDP,  JSON
AT+PRO=3,0   // MQTT, hex
AT+PRO=3,5   // MQTT, JSON
AT+PRO=4,5   // TCP,  JSON
```

Voor MQTT stel je verder in (sectie 3.3): `AT+SERVADDR=<broker>,<poort>`, `AT+CLIENT=<client-id>`, `AT+UNAME` / `AT+PWD`, `AT+PUBTOPIC=<topic>`, `AT+SUBTOPIC=<topic>`, `AT+MQOS=<0|1|2>`, `AT+TLSMOD` (versleuteling), `AT+TDC=600` (zendinterval 10 min) en `AT+APN=<apn van je operator>`.

Dit is in één toestel het verschil tussen de vier protocollen van de les:

| Protocol | Transport | Model | Wanneer |
|---|---|---|---|
| **MQTT** | TCP | publiceren/abonneren via broker | meerdere afnemers, dashboard, betrouwbare aflevering (QoS) |
| **CoAP** | UDP | vraag/antwoord, "HTTP voor kleine toestellen" | minimaal energie- en dataverbruik |
| **UDP** | — | ruwe datagrammen | eigen server, geen overhead, geen garantie |
| **TCP** | — | ruwe verbinding | eigen server, wel aflevergarantie |

Met `…,5` stuurt de sensor **JSON** in plaats van hex. Dat is groter (honderden bytes i.p.v. 11), maar NB-IoT kan dat aan en je hoeft niets meer te decoderen. De handleiding geeft dit voorbeeld (ingekort):

In [ ]:
import json, pandas as pd

voorbeeld_json = '''{"IMEI":"868508065628110","IMSI":"460240210507481","Model":"S31x-CB",
"temperature":25.6,"humidity":52.5,"interrupt":0,"interrupt_level":0,"count":0,
"battery":3.48,"signal":21,"time":"2024/12/03 03:24:54",
"latitude":0.000000,"longitude":0.000000,"gps_time":"1970-01-01T00:00:00Z",
"1":[24.2,52.5,"2024-12-02T05:34:19Z"],"2":[24.1,52.5,"2024-12-02T05:19:19Z"],
"3":[24.1,52.6,"2024-12-02T05:04:19Z"],"4":[24.2,52.8,"2024-12-02T04:49:19Z"],
"5":[24.3,52.0,"2024-12-02T04:34:19Z"],"6":[24.2,51.6,"2024-12-02T04:19:19Z"],
"7":[24.5,50.2,"2024-12-02T03:39:02Z"],"8":[24.3,50.1,"2024-12-02T03:24:02Z"]}'''

bericht = json.loads(voorbeeld_json)
print("Actuele meting:", bericht["temperature"], "°C,", bericht["humidity"], "% RV,",
      "batterij", bericht["battery"], "V, signaal", bericht["signal"])

# De sleutels "1".."8" zijn de laatste 8 metingen (datalog): [temperatuur, RV, tijdstip]
historiek = pd.DataFrame([bericht[str(i)] for i in range(1, 9)],
                         columns=["temperatuur_C", "RV_pct", "tijd"])
historiek["tijd"] = pd.to_datetime(historiek["tijd"])
historiek = historiek.sort_values("tijd")
historiek

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(8, 3.5))
ax1.plot(historiek["tijd"], historiek["temperatuur_C"], marker="o", label="temperatuur (°C)")
ax1.set_ylabel("°C")
ax2 = ax1.twinx()
ax2.plot(historiek["tijd"], historiek["RV_pct"], marker="s", color="tab:gray", label="RV (%)")
ax2.set_ylabel("% RV")
ax1.set_title("Datalog uit één S31-CB-uplink (voorbeeld uit de handleiding)")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

### Opdracht 4.1
Welke AT-commando's zou je naar de S31-CB sturen om elke 10 minuten via MQTT een JSON-bericht te publiceren op `broker.hivemq.com`, topic `ugent/i700301/stal/afdeling3`? Schrijf ze uit (zie de lijst hierboven; de exacte syntaxis staat in §3.3 van de handleiding).

### Opdracht 4.2
De LoRaWAN-versie stuurt 11 bytes, de NB-IoT-versie in JSON-modus enkele honderden. Geef twee redenen waarom LoRaWAN zo zuinig *moet* zijn en NB-IoT dat niet hoeft. (Tip: denk aan de *duty cycle* op 868 MHz en aan wat je per maand betaalt.)

<details>
<summary><b>Oplossing tonen</b></summary>

**4.1**
```
AT+PRO=3,5
AT+SERVADDR=broker.hivemq.com,1883
AT+CLIENT=s31cb-afdeling3
AT+PUBTOPIC=ugent/i700301/stal/afdeling3
AT+MQOS=1
AT+TDC=600
AT+APN=<apn van de operator>
```
Geen `AT+UNAME`/`AT+PWD`: de publieke broker vraagt geen aanmelding. Bij een eigen broker wel, plus `AT+TLSMOD` voor versleuteling.

**4.2** (1) Op 868 MHz geldt een wettelijke *duty cycle* van 1 %: hoogstens 36 s zendtijd per uur, en met 0,3–50 kbit/s kost elke byte veel zendtijd; TTN hanteert bovendien een fair-use-beleid van 30 s uplink per dag. (2) Zenden kost energie: 11 bytes = milliseconden, dus jaren op één batterij. NB-IoT gebruikt gelicentieerd spectrum zonder duty cycle, haalt tientallen kbit/s, en je betaalt per simkaart per maand (binnen een bundel), niet per byte: 300 bytes JSON zijn dan geen probleem, al kost het wel meer batterij.

</details>

## Terugkoppeling naar de les

| Slide | Wat je hier concreet zag |
|---|---|
| Verbindingstypes | LoRaWAN (S31-LB) · Bluetooth (configuratie §1.6) · ethernet/wifi/4G (uplink van de gateway) · NB-IoT/LTE-M (S31-CB) |
| Protocollen | MQTT, CoAP, UDP, TCP kiesbaar met `AT+PRO`; HTTP via de integraties van TTN/Datacake; Modbus via de zustermodule **RS485-LB**, die een bestaande Modbus-sensor of klimaatcomputer uitleest en via LoRaWAN doorstuurt |

Ga verder met `02_mqtt_live_meekijken.ipynb`: daar laten we een virtuele S31-CB exact dit JSON-formaat live publiceren.